# 多级索引与高级选择

学习目标：用多个标签组织记录，完成截面与范围选择、层级调整和按层级对齐，并检查排序与缺失标签的边界。

前置知识：MultiIndex 的层级与元组标签、标签选择、排序与重塑。

运行环境：Python 3.12、pandas 3.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例输入就地构造，明确说明复用的小表；后续单元沿用首次导入的 pd。主线中的日期是格式统一的文本标签，不进行时间解析。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按地区与日期查询

一张每日销量表用“地区、日期”共同标识记录。MultiIndex 把多个层级组成一个轴上的标签；这里每个完整行标签是一个二元组，DataFrame 仍然是二维表格。

from_tuples 适合已有逐条组合标签的输入，names 给各层命名。下面先读取华东某天的销量，再取华东全部日期；units 的单位为件。

In [1]:
import pandas as pd

index = pd.MultiIndex.from_tuples(
    [
        ("华东", "2026-09-01"),
        ("华东", "2026-09-02"),
        ("华北", "2026-09-01"),
        ("华北", "2026-09-02"),
    ],
    names=["region", "date"],
)
sales = pd.DataFrame({"units": [10, 14, 20, 24]}, index=index)

print(sales)  # 四行一列；两层行标签不算数据列，units 为 int64。
print(sales.loc[("华东", "2026-09-02"), "units"])  # 14。
east = sales.loc["华东", :]
print(east)  # 10、14 两行；只剩 date 这一层索引。
print(east.shape, east.index.name)  # (2, 1) date。

                   units
region date             
华东     2026-09-01     10
       2026-09-02     14
华北     2026-09-01     20
       2026-09-02     24
14
            units
date             
2026-09-01     10
2026-09-02     14
(2, 1) date


## 2 完整键、部分键与截面

完整元组按层级顺序指定一个键，部分键只给出前面的层级。上例的“华东”是第一层的部分键；不能把第二层日期直接当作第一层标签查找。

xs 表示截面选择，可以通过 level 指定任一层。它默认去掉选中的层；drop_level=False 保留原有层级。下面继续使用 sales，比较同一天的两个地区。xs 用于读取，修改原表应使用 loc。

In [2]:
daily = sales.xs("2026-09-02", level="date")
daily_kept = sales.xs("2026-09-02", level="date", drop_level=False)

print(daily)  # 华东 14、华北 24，索引名为 region。
print(daily_kept.index.tolist())
# [('华东', '2026-09-02'), ('华北', '2026-09-02')]。
print(daily.shape, daily_kept.shape)  # 都为 (2, 1)，保留层级不增加行数。
print(daily_kept.index.names)  # ['region', 'date']。

        units
region       
华东         14
华北         24
[('华东', '2026-09-02'), ('华北', '2026-09-02')]
(2, 1) (2, 1)
['region', 'date']


## 3 根据输入形式构造索引

已有两列平行标签时，用 from_arrays：每个数组表示一层，各位置共同组成一条标签。已有标签表时，用 from_frame：每列表示一层，默认沿用列名作为层名。

这两种输入都按已有记录构造，不补出没有提供的组合。

In [3]:
regions = ["华东", "华东", "华北"]
dates = ["2026-09-01", "2026-09-02", "2026-09-01"]
from_arrays = pd.MultiIndex.from_arrays(
    [regions, dates], names=["region", "date"]
)
keys = pd.DataFrame({"region": regions, "date": dates})
from_frame = pd.MultiIndex.from_frame(keys)

print(from_arrays.tolist())  # 三个组合；没有华北 2026-09-02。
print(from_frame.names)  # ['region', 'date']，来自列名。
print(from_arrays.equals(from_frame))  # True，标签与顺序相同。

[('华东', '2026-09-01'), ('华东', '2026-09-02'), ('华北', '2026-09-01')]
['region', 'date']
True


需要列出每个地区与每个日期的全部组合时，用 from_product。各输入应提供该层的不重复标签；两层各有两个标签时得到四个组合。

它只创建索引，不代表每个组合都有实际观测。下面沿用本节三行 keys，重建到完整组合时，未观测组合的销量保留缺失。

In [4]:
complete_index = pd.MultiIndex.from_product(
    [["华东", "华北"], ["2026-09-01", "2026-09-02"]],
    names=["region", "date"],
)
observed = pd.DataFrame({"units": [10, 14, 20]}, index=from_frame)
complete = observed.reindex(complete_index)

print(complete)  # 华北 2026-09-02 为 NaN，其余三行保留原值。
print(complete.shape, complete["units"].dtype)  # (4, 1) float64。
print(complete.index.is_unique)  # True，检查的是完整组合键。

                   units
region date             
华东     2026-09-01   10.0
       2026-09-02   14.0
华北     2026-09-01   20.0
       2026-09-02    NaN
(4, 1) float64
True


## 4 层级、编码与实际标签

names 是层名，nlevels 是层数。levels 保存每层的标签集合，codes 保存各行引用这些标签的位置；levels 并不是逐行标签。需要每行某一层的实际值时，用 get_level_values。

下面继续查看开篇的 sales。同一地区在多行出现是正常的；是否唯一要检查完整元组，而不是某一层是否重复。

下图用地区和星期构造一个独立的三行索引，说明编码如何还原元组；它不复用 sales 的日期标签。

![下图用地区和星期构造一个独立的三行索引，说明编码如何还原元组；它不复用 sales 的日期标签。](image/14-levels-codes.png)

In [5]:
print(sales.index.names, sales.index.nlevels)  # ['region', 'date']，2 层。
print(sales.index.levels)  # 每层各两个不同标签。
print(sales.index.codes)  # 本例为 [[0, 0, 1, 1], [0, 1, 0, 1]]。
print(sales.index.get_level_values("region").tolist())
# ['华东', '华东', '华北', '华北']，长度等于行数。

repeated = pd.MultiIndex.from_tuples([("华东", "01"), ("华东", "01")])
print(repeated.is_unique)  # False；构造 MultiIndex 不会自动禁止重复完整键。

['region', 'date'] 2
[['华东', '华北'], ['2026-09-01', '2026-09-02']]
[[0, 0, 1, 1], [0, 1, 0, 1]]
['华东', '华东', '华北', '华北']
False


选择子集后，levels 可能仍保留未使用的标签。判断当前出现了哪些标签，应查看 get_level_values 的结果；需要清理未使用的层标签时，可以调用 remove_unused_levels，它不删除数据行。

下面从 sales 取华东两行并保留两层索引。

In [6]:
east_kept = sales.xs("华东", level="region", drop_level=False)
compact_index = east_kept.index.remove_unused_levels()

print(east_kept.index.levels[0].tolist())  # ['华东', '华北']。
print(east_kept.index.get_level_values("region").unique().tolist())  # ['华东']。
print(compact_index.levels[0].tolist())  # ['华东']。
print(compact_index.equals(east_kept.index))  # True，逐行标签和顺序未变。

['华东', '华北']
['华东']
['华东']
True


层标签缺失时，对应 code 使用 -1；它是缺失标记，不能按普通 Python 负索引把它解释成 levels 的最后一个值。读取实际标签优先使用 get_level_values。

普通整数层含缺失时，提取出的 Index 可能转成 float64 来容纳 NaN。下面只观察这一类输入，不把 dtype 推断推广到所有扩展类型。

In [7]:
missing_index = pd.MultiIndex.from_arrays(
    [[1, None, 2], ["A", "B", "A"]], names=["store", "batch"]
)
store_labels = missing_index.get_level_values("store")

print(missing_index.levels[0].tolist())  # [1, 2]。
print(missing_index.codes[0].tolist())  # [0, -1, 1]；中间一行缺失。
print(store_labels)  # Index([1.0, nan, 2.0], dtype='float64', name='store')。
print(store_labels.isna().tolist())  # [False, True, False]。

[1, 2]
[0, -1, 1]
Index([1.0, nan, 2.0], dtype='float64', name='store')
[False, True, False]


## 5 元组与列表的选择范围

在 loc 中，一个完整元组表示跨层级的一个键，元组列表表示多个完整键；元组里的各层列表则分别限制该层允许的标签。这两种多项选择不能混为一谈。

下面继续使用 sales。显式写出行选择器和列选择器，避免逗号被误解为切换行列轴。完整键外再加列表还可保留 DataFrame 的形状。

In [8]:
one_row = sales.loc[[("华东", "2026-09-01")], :]
chosen_pairs = sales.loc[
    [("华东", "2026-09-01"), ("华北", "2026-09-02")], :
]
chosen_levels = sales.loc[
    (["华东", "华北"], ["2026-09-01", "2026-09-02"]), :
]

print(one_row.shape)  # (1, 1)，仍保留两层索引。
print(chosen_pairs)  # 只取两个明确的组合：10、24。
print(chosen_levels.shape)  # (4, 1)，本表中满足各层条件的四行。
print(chosen_pairs.index.tolist())  # 顺序与给出的两个完整键一致。

(1, 1)
                   units
region date             
华东     2026-09-01     10
华北     2026-09-02     24
(4, 1)
[('华东', '2026-09-01'), ('华北', '2026-09-02')]


## 6 范围切片与排序条件

多级标签按从外层到内层的顺序比较。跨两个层级的范围切片需要相应层级已排序；最直接的准备方法是 sort_index。范围两端按标签切片规则包含在内。

未排序并不等于任何选择都失败。下面的完整键唯一，精确选择可直接成功；同一输入的范围切片会触发 UnsortedIndexError。这里直接观察该异常，再排序完成查询。

In [9]:
unsorted_sales = sales.iloc[[3, 0, 2, 1]]
print(unsorted_sales.index.is_monotonic_increasing)  # False。
print(unsorted_sales.loc[("华东", "2026-09-02"), "units"])  # 14。

# 预期 UnsortedIndexError：MultiIndex 尚未按所需层级排序，不能按这个二层键区间切片。
unsorted_sales.loc[("华东", "2026-09-02"):("华北", "2026-09-01"), :]

False
14


UnsortedIndexError: 'Key length (2) was greater than MultiIndex lexsort depth (0)'

In [10]:
ordered = unsorted_sales.sort_index()
selected = ordered.loc[("华东", "2026-09-02"):("华北", "2026-09-01"), :]
print(selected)  # 两端均保留：华东第二天 14、华北第一天 20。
print(selected.shape, ordered.index.is_monotonic_increasing)  # (2, 1) True。

                   units
region date             
华东     2026-09-02     14
华北     2026-09-01     20
(2, 1) True


sort_index 的 level 可指定排序层，sort_remaining 默认还会排序剩余层。只排序外层并显式关闭 sort_remaining，不保证两层键整体有序。

下面特意让每个地区内部的日期逆序。索引的层级顺序始终是 region、date；按某层排序与交换层级是不同操作。

In [11]:
reverse_dates = sales.iloc[[1, 0, 3, 2]]
outer_only = reverse_dates.sort_index(level="region", sort_remaining=False)
fully_sorted = reverse_dates.sort_index(level="region", sort_remaining=True)

print(outer_only.index.tolist())  # 各地区内部仍是 09-02、09-01。
print(outer_only.index.is_monotonic_increasing)  # False。
print(fully_sorted.index.is_monotonic_increasing)  # True。
print(fully_sorted.index.names)  # ['region', 'date']，没有交换层级。

[('华东', '2026-09-02'), ('华东', '2026-09-01'), ('华北', '2026-09-02'), ('华北', '2026-09-01')]
False
True
['region', 'date']


## 7 交换和重排层级

希望每天的各地区记录放在一起时，先用 swaplevel 交换 region、date，再用 sort_index 排列记录。swaplevel 只交换标签中两个层的位置，不自动排序数据行；默认交换最后两层。

下面继续使用 sales，核对交换前后销量仍与同一地区和日期对应。

In [12]:
swapped = sales.swaplevel("region", "date", axis="index")
by_date = swapped.sort_index()

print(swapped.index.names)  # ['date', 'region']。
print(swapped["units"].tolist())  # [10, 14, 20, 24]，行位置未重新排列。
print(swapped.index.is_monotonic_increasing)  # False。
print(by_date)  # 第一日 10、20，第二日 14、24。
print(by_date.loc[("2026-09-02", "华东"), "units"])  # 14，标签顺序随层级改变。
assert by_date.swaplevel("date", "region").sort_index().equals(sales)

['date', 'region']
[10, 14, 20, 24]
False
                   units
date       region       
2026-09-01 华东         10
           华北         20
2026-09-02 华东         14
           华北         24
14


三个以上层级需要一次改变顺序时，用 reorder_levels 指定完整的新顺序。它不能省略或重复层级，也不自动排序行；名称和从 0 开始的层位置都可用于指定层级。

下面新增渠道标签，改为先看日期，再看地区与渠道。

In [13]:
channel_index = pd.MultiIndex.from_tuples(
    [
        ("华东", "门店", "2026-09-02"),
        ("华东", "线上", "2026-09-01"),
        ("华北", "门店", "2026-09-01"),
    ],
    names=["region", "channel", "date"],
)
channel_sales = pd.DataFrame({"units": [14, 8, 20]}, index=channel_index)
reordered = channel_sales.reorder_levels(["date", "region", "channel"])

print(reordered.index.names)  # ['date', 'region', 'channel']。
print(reordered["units"].tolist())  # [14, 8, 20]，仅换层级时行位置不变。
print(reordered.sort_index())  # 排序后销量为 8、20、14；三个层级都保留。

['date', 'region', 'channel']
[14, 8, 20]
                           units
date       region channel       
2026-09-01 华东     线上           8
           华北     门店          20
2026-09-02 华东     门店          14


## 8 按索引层级聚合

groupby 的 level 按索引层分组，不必先把标签转回普通列。下面继续使用 sales，分别统计地区总量与日均销量；聚合后每个地区只剩一行。

本例没有分类标签或缺失键，仍显式给出 observed 与 dropna，以便看清分组约定。需要把缺失标签也作为一组时，应设 dropna=False。

In [14]:
groups = sales.groupby(level="region", sort=True, observed=True, dropna=False)
totals = groups.sum()
means = groups.mean()

print(totals)  # 华东 24、华北 44；两行一列，region 为普通单层索引。
print(means)  # 华东 12.0、华北 22.0；units 为 float64。
print(totals.shape, totals.index.name)  # (2, 1) region。
assert totals["units"].sum() == sales["units"].sum()

        units
region       
华东         24
华北         44
        units
region       
华东       12.0
华北       22.0
(2, 1) region


缺失标签是否参与统计会改变总量。下面有一条地区未填的观测，销量本身并不缺失；它是否被分组保留由 dropna 决定。

In [15]:
incomplete_index = pd.MultiIndex.from_tuples(
    [("华东", "2026-09-01"), (None, "2026-09-01")],
    names=["region", "date"],
)
incomplete = pd.DataFrame({"units": [10, 5]}, index=incomplete_index)
without_missing = incomplete.groupby(level="region", dropna=True).sum()
with_missing = incomplete.groupby(level="region", dropna=False).sum()

print(without_missing)  # 只有华东 10。
print(with_missing)  # 增加 NaN 标签组，销量为 5。
print(with_missing["units"].sum())  # 15，保留了两条观测的总量。

        units
region       
华东         10
        units
region       
华东         10
NaN         5
15


## 9 按层级对齐统计结果

把地区均值对应回各天记录时，可用 reindex 的 level：按指定层的标签匹配，把同一地区的均值重复到该地区各行。它不会把两个地区的均值按位置机械重复。

下面复用 sales 和上节 means，先把均值顺序颠倒，再对齐并计算偏离地区均值的件数。

In [16]:
reversed_means = means.iloc[::-1]
expanded_means = reversed_means.reindex(sales.index, level="region")
deviations = sales - expanded_means

print(expanded_means)  # 按地区匹配：12.0、12.0、22.0、22.0。
print(deviations)  # 各地区依次为 -2.0、2.0，四行一列。
print(expanded_means.index.equals(sales.index))  # True。
print(deviations["units"].dtype)  # float64，与均值运算后的类型。
assert deviations.groupby(level="region")["units"].sum().eq(0).all()

                   units
region date             
华东     2026-09-01   12.0
       2026-09-02   12.0
华北     2026-09-01   22.0
       2026-09-02   22.0
                   units
region date             
华东     2026-09-01   -2.0
       2026-09-02    2.0
华北     2026-09-01   -2.0
       2026-09-02    2.0
True
float64


align 可同时返回对齐后的两个对象；axis="index" 指定行轴，level="region" 指定匹配层。对 Series 基准做算术时，sub 也支持相同的层级匹配参数。

下面继续用 sales 与 reversed_means，对照显式展开和直接按层计算。基准缺少某个地区时，对应差值应保留缺失，不能当作零基准。

In [17]:
left, right = sales.align(
    reversed_means, axis="index", level="region", join="left"
)
direct = sales.sub(reversed_means["units"], axis="index", level="region")
print(left.equals(sales), right.equals(expanded_means))  # True True。
print(direct.equals(deviations))  # True，两种写法得到相同值、标签与类型。

east_mean_only = means.loc[["华东"], "units"]
missing_baseline = sales.sub(east_mean_only, axis="index", level="region")
print(missing_baseline)  # 华东为 -2.0、2.0，华北两行均为 NaN。
print(missing_baseline.shape)  # (4, 1)，原有记录仍保留。

True True
True
                   units
region date             
华东     2026-09-01   -2.0
       2026-09-02    2.0
华北     2026-09-01    NaN
       2026-09-02    NaN
(4, 1)


## 10 列轴上的多级索引

MultiIndex 也可以作为列标签。例如把“指标、渠道”共同作为列键；xs 的 axis="columns" 选择列截面，loc 的完整列键仍按层级顺序组成元组。

这时 swaplevel、reorder_levels 和 sort_index 也要显式选择列轴。

In [18]:
columns = pd.MultiIndex.from_tuples(
    [("units", "门店"), ("units", "线上")], names=["metric", "channel"]
)
wide = pd.DataFrame([[10, 8], [20, 6]], index=["华东", "华北"], columns=columns)
shops = wide.xs("门店", level="channel", axis="columns")
by_channel = wide.swaplevel("metric", "channel", axis="columns")

print(shops)  # 华东 10、华北 20；列轴只剩 metric，形状为 (2, 1)。
print(wide.loc[:, ("units", "线上")].tolist())  # [8, 6]。
print(by_channel.columns.names)  # ['channel', 'metric']。
print(by_channel.sort_index(axis="columns").columns.tolist())
# [('线上', 'units'), ('门店', 'units')]，按新的列标签排列。

metric  units
华东         10
华北         20
[8, 6]
['channel', 'metric']
[('线上', 'units'), ('门店', 'units')]


## 11 选学：多层切片写法

slice(None) 表示该层全部标签；给每一层单独写选择条件，可以取所有地区的一段日期。IndexSlice 允许在同一位置使用冒号写法，不改变选择语义或排序要求。

下面使用已排序的 sales。日期文本统一为年、月、日格式，本例可以按文本顺序选择；这里没有启用时间索引的部分日期解析。

In [19]:
selected_plain = sales.loc[
    (slice(None), slice("2026-09-02", "2026-09-02")), :
]
idx = pd.IndexSlice
selected_short = sales.loc[idx[:, "2026-09-02":"2026-09-02"], :]

print(selected_short)  # 华东 14、华北 24；保留两层索引。
print(selected_short.shape)  # (2, 1)。
print(selected_short.equals(selected_plain))  # True。

                   units
region date             
华东     2026-09-02     14
华北     2026-09-02     24
(2, 1)
True


## 12 选学：其他索引类型的入口

多级索引描述的是层级组合；下面这些索引类型描述一层标签的数据性质。选择时先看标签是否有日期、类别或区间含义，而不是一律转成文本。

| 类型 | 中文名称／含义 | 适用入口 |
| --- | --- | --- |
| RangeIndex | 等步长整数范围索引 | 不需要业务标签时的默认行编号 |
| DatetimeIndex | 日期时间索引 | 标签本身是日期时间，需要时间属性与时间选择 |
| CategoricalIndex | 分类索引 | 固定类别集合、重复标签或明确的类别顺序 |
| IntervalIndex | 区间索引 | 标签本身是区间，需要明确开闭端点 |

RangeIndex 的 stop 不包含在范围内。DatetimeIndex 保存日期时间值，与主线使用的日期文本不同；下面仅构造无时区的简单日期，并明确使用微秒单位。

In [20]:
row_index = pd.RangeIndex(start=0, stop=6, step=2, name="row")
date_index = pd.DatetimeIndex(
    ["2026-09-01", "2026-09-02"], dtype="datetime64[us]", name="date"
)

print(row_index.tolist())  # [0, 2, 4]，不包含 6。
print(date_index)  # 两个午夜时间点；dtype 为 datetime64[us]，没有时区。
print(date_index.dtype)  # datetime64[us]；不把所有日期索引都假定为纳秒单位。

[0, 2, 4]
DatetimeIndex(['2026-09-01', '2026-09-02'], dtype='datetime64[us]', name='date', freq=None)
datetime64[us]


CategoricalIndex 保留类别集合与有序属性；下面只使用已定义类别和明确的缺失标签。IntervalIndex 的区间采用统一的端点约定，contains 逐个判断给定值落在哪个区间中。

In [21]:
grades = pd.CategoricalIndex(
    ["高", "低", None], categories=["低", "中", "高"], ordered=True
)
intervals = pd.IntervalIndex.from_breaks([0, 10, 20], closed="right")

print(grades.tolist())  # ['高', '低', nan]，第三个标签缺失。
print(grades.categories.tolist(), grades.ordered)  # ['低', '中', '高'] True。
print(intervals)  # (0, 10] 与 (10, 20]，左开右闭。
print(intervals.contains(10).tolist())  # [True, False]，10 只属于第一个区间。

['高', '低', nan]


['低', '中', '高'] True
IntervalIndex([(0, 10], (10, 20]], dtype='interval[int64, right]')
[True, False]


## 本章小结

（1）MultiIndex 用多个层级共同表示一个轴上的标签。完整键、部分键和各层条件的选择范围不同，选择后还需检查是否保留层级。

（2）levels 与 codes 描述层标签及引用关系；查看逐行标签用 get_level_values。缺失 code 为 -1，完整键是否唯一要另行检查。

（3）swaplevel 和 reorder_levels 改变层的位置，sort_index 排列记录。精确选择成功不能证明范围切片的排序条件已满足。

（4）groupby(level=...) 按层聚合；reindex、align 或算术方法的 level 参数可把结果按指定层对应回记录。核对标签、顺序、缺失和结果形状。

## 练习

（1）为下面三条观测建立“地区、日期”多级索引，读取华北全部日期。随后要求改变为“报告必须包含两个地区与两个日期的全部组合”，选择合适的构造与重建方法，解释未观测的组合为什么不应自动填零。

In [22]:
records = pd.DataFrame(
    {
        "region": ["华东", "华北", "华北"],
        "date": ["2026-09-01", "2026-09-01", "2026-09-02"],
        "units": [11, 21, 25],
    }
)
# 在此先构造三行观测的索引，再扩展到四个组合。
# 检查：华东 2026-09-02 的 units 缺失，已有三条销量不变。

（2）预测下面三个结果各有几行、保留几层索引，再运行核对。比较完整键列表与每层列表的差异，解释为什么选择条件看似相近却可能得到不同的行数。

In [23]:
exercise_index = pd.MultiIndex.from_product(
    [["华东", "华北"], ["01", "02"]], names=["region", "day"]
)
exercise = pd.DataFrame({"units": [1, 2, 3, 4]}, index=exercise_index)
a = exercise.loc[[("华东", "01"), ("华北", "02")], :]
b = exercise.loc[(["华东", "华北"], ["01", "02"]), :]
c = exercise.xs("02", level="day")

print(a)
print(b)
print(c)
# 运行后比较 shape、index.names 和 index.tolist()，再核对自己的预测。

            units
region day       
华东     01       1
华北     02       4

            units
region day       
华东     01       1
       02       2
华北     01       3
       02       4
        units
region       
华东          2
华北          4


（3）把下表改成“日期、地区”的层级顺序，并按新标签排序，取 01 到 02 的日期范围。新增约束为“必须保留输入行顺序，只查华东的 02”，是否还需要排序？说明范围选择与完整键精确选择的区别。

In [24]:
exercise_index = pd.MultiIndex.from_tuples(
    [("华北", "02"), ("华东", "01"), ("华北", "01"), ("华东", "02")],
    names=["region", "day"],
)
exercise = pd.DataFrame({"units": [24, 10, 20, 14]}, index=exercise_index)
# 在此交换层级并排序，打印结果；另从原表精确选择一条记录。
# 检查：排序后的销量依次为 10、20、14、24；精确选择值为 14。

（4）按地区求均值，并按层级对应回每一行后计算差值。随后只保留华东的均值作为基准，预测华北两条差值的结果；解释为何不能把这一个均值按位置重复给所有行。

In [25]:
exercise_index = pd.MultiIndex.from_product(
    [["华东", "华北"], ["01", "02"]], names=["region", "day"]
)
exercise = pd.DataFrame({"units": [8, 12, 18, 26]}, index=exercise_index)
# 在此按 region 聚合，再用 level 参数完成对齐与减法。
# 检查：完整基准下每组差值之和为 0，结果保留四条组合标签。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [MultiIndex / advanced indexing](https://pandas.pydata.org/docs/user_guide/advanced.html) 的 Basic indexing、Advanced indexing、Sorting a MultiIndex 与 Advanced reindexing and alignment；[MultiIndex](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.html) 的层级结构；[from_tuples](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_tuples.html)、[from_arrays](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_arrays.html)、[from_frame](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_frame.html)、[from_product](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_product.html) 的输入与组合范围；[levels](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.levels.html)、[codes](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.codes.html)、[get_level_values](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.get_level_values.html) 的标签、编码与缺失类型；[remove_unused_levels](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.remove_unused_levels.html)、[is_unique](https://pandas.pydata.org/docs/reference/api/pandas.Index.is_unique.html)；[xs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.xs.html) 的 axis、level、drop_level 与读取限制；[swaplevel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.swaplevel.html)、[reorder_levels](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reorder_levels.html)、[sort_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_index.html) 的层级重排和 sort_remaining；[groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) 的 level、dropna、observed；[reindex](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html)、[align](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.align.html)、[sub](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sub.html) 的按层广播与对齐；[IndexSlice](https://pandas.pydata.org/docs/reference/api/pandas.IndexSlice.html) 的切片简写；[RangeIndex](https://pandas.pydata.org/docs/reference/api/pandas.RangeIndex.html)、[DatetimeIndex](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.html)、[CategoricalIndex](https://pandas.pydata.org/docs/reference/api/pandas.CategoricalIndex.html)、[IntervalIndex](https://pandas.pydata.org/docs/reference/api/pandas.IntervalIndex.html) 的类型适用范围；[IntervalIndex.from_breaks](https://pandas.pydata.org/docs/reference/api/pandas.IntervalIndex.from_breaks.html)、[contains](https://pandas.pydata.org/docs/reference/api/pandas.IntervalIndex.contains.html) 的端点与成员判断。 |
| GitHub 官方项目（版本化来源） | [pandas/core/indexes/multi.py（v3.0.5）](https://github.com/pandas-dev/pandas/blob/v3.0.5/pandas/core/indexes/multi.py) 的 MultiIndex 中 &#95;validate&#95;codes 与 &#95;verify&#95;integrity：缺失层标签对应 code -1；同时核对当前安装的 pandas 3.0.6 同名实现。 pandas v3.0.6 文档源码：[advanced](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/advanced.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |